In [26]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE_DIR_ROOT = r"C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV": "prune_layers_CONV",
    "FHL": "prune_layers_FHL",
    "SHL": "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL": "prune_layers_ALL"
}

BATCH_SIZES = [64, 1024]
PRUNING_PERCENTAGES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# =========================
# OUTPUT
# =========================
AUC_GRAPH_DIR = os.path.join(BASE_DIR_ROOT, "AUC_graph_0-100")
AUC_DATA_DIR = os.path.join(BASE_DIR_ROOT, "AUC_data_0-100")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10
})

CE_MAX = np.log(10)
X_CUTOFF = 400

# =========================
# COLORS
# =========================
PRUNING_COLOR_MAP = {
    0.0: "#17becf",
    0.1: "#B9D9EB",
    0.2: "#bcbd22",
    0.3: "#7f7f7f",
    0.4: "#e377c2",
    0.5: "#8c564b",
    0.6: "#800080",
    0.7: "#d62728",
    0.8: "#2ca02c",
    0.9: "#C5B0D5",
    1.0: "#1f77b4"
}

# =========================
# HELPERS
# =========================
def fmt_p(p):
    s = f"{p:.3f}".rstrip("0").rstrip(".")
    if "." not in s:
        s += ".0"
    return s

def label_from_p(p):
    pct = p * 100
    if abs(pct - round(pct)) < 1e-9:
        return f"P%={int(round(pct))}"
    return f"P%={pct:g}"

all_auc_records = []

# =========================
# MAIN LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:
    print(f"\nProcessing prune layer: {prune_layer}")

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])

    layer_graph_dir = os.path.join(AUC_GRAPH_DIR, prune_layer)
    layer_data_dir = os.path.join(AUC_DATA_DIR, prune_layer)
    os.makedirs(layer_graph_dir, exist_ok=True)
    os.makedirs(layer_data_dir, exist_ok=True)

    for bs in BATCH_SIZES:
        avg_dfs = {}

        for p in PRUNING_PERCENTAGES:
            p_str = fmt_p(p)
            file_path = os.path.join(
                base_dir,
                f"p-percentage_{p_str}",
                f"batch_size_{bs}",
                f"averaged_runs_conv_{prune_layer}_p_{p_str}_bs_{bs}.csv"
            )

            if not os.path.isfile(file_path):
                print(f"[WARNING] Missing file: {file_path}")
                continue

            avg_dfs[p] = pd.read_csv(file_path)

        if not avg_dfs:
            print(f"[WARNING] No data found for {prune_layer}, BS={bs}")
            continue

        # ---------------------------------
        # SHORTEST-CURVE LOGIC
        # ---------------------------------
        non100_dfs = {
            p: df[df["Batch_Number"] <= X_CUTOFF].copy()
            for p, df in avg_dfs.items()
            if p != 1.0 and p!=0.995 and "Batch_Number" in df.columns
        }
        non100_dfs = {p: df for p, df in non100_dfs.items() if not df.empty}

        if non100_dfs:
            lowest_max_batch = min(df["Batch_Number"].max() for df in non100_dfs.values())
        else:
            all_trunc = {
                p: df[df["Batch_Number"] <= X_CUTOFF].copy()
                for p, df in avg_dfs.items()
                if "Batch_Number" in df.columns
            }
            all_trunc = {p: df for p, df in all_trunc.items() if not df.empty}
            if not all_trunc:
                print(f"[WARNING] No valid Batch_Number data for {prune_layer}, BS={bs}")
                continue
            lowest_max_batch = min(df["Batch_Number"].max() for df in all_trunc.values())

        def plot_and_save_ce(ce_column, y_label, file_stub):
            plt.figure(figsize=(10, 5))
            plt.title(f"CNN-{prune_layer} | CIFAR-10 | BS={bs}")
            plt.xlabel("Batch Number")
            plt.ylabel(y_label)
            plt.xlim(0, lowest_max_batch)
            plt.xticks(np.arange(0, int(lowest_max_batch) + 1, 50))
            plt.yticks(np.arange(0, 2.5, 0.5))
            plt.ylim(0, 2.5)

            plt.text(
                0.40,
                CE_MAX,
                r"$\ln(10)$",
                transform=plt.gca().get_yaxis_transform(),
                fontsize=12,
                va="center",
                ha="left",
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1)
            )

            for p in sorted(avg_dfs.keys(), reverse=True):
                df = avg_dfs[p].copy()
                df = df[df["Batch_Number"] <= min(X_CUTOFF, lowest_max_batch)]

                color = PRUNING_COLOR_MAP.get(p, "#000000")

                if p == 1.0:
                    ce_value = min(df[ce_column].iloc[-1], CE_MAX)
                    x = np.array([0, lowest_max_batch])
                    y = np.array([ce_value, ce_value])
                    auc = ce_value * lowest_max_batch

                    plt.plot(
                        x, y,
                        color=color,
                        linewidth=2.5,
                        label=f"{label_from_p(p)} | AUC={auc:.2f}"
                    )
                else:
                    x = df["Batch_Number"].to_numpy()
                    y = np.minimum(df[ce_column].to_numpy(), CE_MAX)

                    mask = np.isfinite(x) & np.isfinite(y)
                    x, y = x[mask], y[mask]

                    if len(x) == 0:
                        continue

                    auc = np.trapezoid(y, x)

                    plt.plot(
                        x, y,
                        color=color,
                        linewidth=2.0,
                        label=f"{label_from_p(p)} | AUC={auc:.2f}"
                    )
                    plt.fill_between(x, y, alpha=0.2, color=color)

                all_auc_records.append([prune_layer, bs, p, ce_column, auc])

            plt.grid(True)
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
            plt.tight_layout()

            fig_path = os.path.join(layer_graph_dir, f"{file_stub}_BS_{bs}.png")
            plt.savefig(fig_path, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"[SAVED] {fig_path}")

        plot_and_save_ce("Avg_CE_Train", "Average CE_Train", "Avg_CE_Train")
        plot_and_save_ce("Avg_CE_Test", "Average CE_Test", "Avg_CE_Test")

if all_auc_records:
    auc_df = pd.DataFrame(
        all_auc_records,
        columns=["Prune_Layer", "Batch_Size", "Pruning_Percentage", "CE_Type", "AUC"]
    )
    csv_path = os.path.join(AUC_DATA_DIR, "AUC_ALL_BS.csv")
    auc_df.to_csv(csv_path, index=False)
    print(f"[SAVED] {csv_path}")

print("\nAll AUC graphs and combined data saved successfully.")


Processing prune layer: CONV
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\AUC_graph_0-100\CONV\Avg_CE_Train_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\AUC_graph_0-100\CONV\Avg_CE_Test_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\AUC_graph_0-100\CONV\Avg_CE_Train_BS_1024.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\AUC_graph_0-100\CONV\Avg_CE_Test_BS_1024.png

Processing prune layer: FHL
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\AUC_graph_0-100\FHL\Avg_CE_Train_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\AUC_graph_0-100\FHL\Avg_CE_Test_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neu

In [25]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE_DIR_ROOT = r"C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV": "prune_layers_CONV",
    "FHL": "prune_layers_FHL",
    "SHL": "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL": "prune_layers_ALL"
}

BATCH_SIZES = [64, 1024]
PRUNING_PERCENTAGES = [0.9, 0.92, 0.94, 0.96, 0.98, 0.985, 0.99, 0.995,1.0]

# =========================
# OUTPUT
# =========================
AUC_GRAPH_DIR = os.path.join(BASE_DIR_ROOT, "AUC_graph_90-100")
AUC_DATA_DIR = os.path.join(BASE_DIR_ROOT, "AUC_data_90-100")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10
})

CE_MAX = np.log(10)
X_CUTOFF = 400

# =========================
# COLORS
# =========================
PRUNING_COLOR_MAP = {
    0.9:   "#C5B0D5",
    0.92:  "#8DD3C7",
    0.94:  "#BEBADA",
    0.96:  "#80B1D3",
    0.98:  "#FDB462",
    0.985: "#6A5ACD",
    0.99:  "#FF6F61",
    0.995: "#00A86B",
    1.0:   "#1f77b4"
}

# =========================
# HELPERS
# =========================
def fmt_p(p):
    s = f"{p:.3f}".rstrip("0").rstrip(".")
    if "." not in s:
        s += ".0"
    return s

def label_from_p(p):
    pct = p * 100
    if abs(pct - round(pct)) < 1e-9:
        return f"P%={int(round(pct))}"
    return f"P%={pct:g}"

all_auc_records = []

# =========================
# MAIN LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:
    print(f"\nProcessing prune layer: {prune_layer}")

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])

    layer_graph_dir = os.path.join(AUC_GRAPH_DIR, prune_layer)
    layer_data_dir = os.path.join(AUC_DATA_DIR, prune_layer)
    os.makedirs(layer_graph_dir, exist_ok=True)
    os.makedirs(layer_data_dir, exist_ok=True)

    for bs in BATCH_SIZES:
        avg_dfs = {}

        for p in PRUNING_PERCENTAGES:
            p_str = fmt_p(p)
            file_path = os.path.join(
                base_dir,
                f"p-percentage_{p_str}",
                f"batch_size_{bs}",
                f"averaged_runs_conv_{prune_layer}_p_{p_str}_bs_{bs}.csv"
            )

            if not os.path.isfile(file_path):
                print(f"[WARNING] Missing file: {file_path}")
                continue

            avg_dfs[p] = pd.read_csv(file_path)

        if not avg_dfs:
            print(f"[WARNING] No data found for {prune_layer}, BS={bs}")
            continue

        non100_dfs = {
            p: df[df["Batch_Number"] <= X_CUTOFF].copy()
            for p, df in avg_dfs.items()
            if p != 1.0 and p!=0.995 and "Batch_Number" in df.columns
        }
        non100_dfs = {p: df for p, df in non100_dfs.items() if not df.empty}

        if non100_dfs:
            lowest_max_batch = min(df["Batch_Number"].max() for df in non100_dfs.values())
        else:
            all_trunc = {
                p: df[df["Batch_Number"] <= X_CUTOFF].copy()
                for p, df in avg_dfs.items()
                if "Batch_Number" in df.columns
            }
            all_trunc = {p: df for p, df in all_trunc.items() if not df.empty}
            if not all_trunc:
                print(f"[WARNING] No valid Batch_Number data for {prune_layer}, BS={bs}")
                continue
            lowest_max_batch = min(df["Batch_Number"].max() for df in all_trunc.values())

        def plot_and_save_ce(ce_column, y_label, file_stub):
            plt.figure(figsize=(10, 5))
            plt.title(f"CNN-{prune_layer} | CIFAR-10 | BS={bs}")
            plt.xlabel("Batch Number")
            plt.ylabel(y_label)
            plt.xlim(0, lowest_max_batch)
            plt.xticks(np.arange(0, int(lowest_max_batch) + 1, 50))
            plt.yticks(np.arange(0, 2.5, 0.5))
            plt.ylim(0, 2.5)

            plt.text(
                0.40,
                CE_MAX,
                r"$\ln(10)$",
                transform=plt.gca().get_yaxis_transform(),
                fontsize=12,
                va="center",
                ha="left",
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1)
            )

            for p in sorted(avg_dfs.keys(), reverse=True):
                df = avg_dfs[p].copy()
                df = df[df["Batch_Number"] <= min(X_CUTOFF, lowest_max_batch)]

                color = PRUNING_COLOR_MAP.get(p, "#000000")

                if p == 1.0:
                    ce_value = min(df[ce_column].iloc[-1], CE_MAX)
                    x = np.array([0, lowest_max_batch])
                    y = np.array([ce_value, ce_value])
                    auc = ce_value * lowest_max_batch

                    plt.plot(
                        x, y,
                        color=color,
                        linewidth=2.5,
                        label=f"{label_from_p(p)} | AUC={auc:.2f}"
                    )
                else:
                    x = df["Batch_Number"].to_numpy()
                    y = np.minimum(df[ce_column].to_numpy(), CE_MAX)

                    mask = np.isfinite(x) & np.isfinite(y)
                    x, y = x[mask], y[mask]

                    if len(x) == 0:
                        continue

                    auc = np.trapezoid(y, x)

                    plt.plot(
                        x, y,
                        color=color,
                        linewidth=2.0,
                        label=f"{label_from_p(p)} | AUC={auc:.2f}"
                    )
                    plt.fill_between(x, y, alpha=0.2, color=color)

                all_auc_records.append([prune_layer, bs, p, ce_column, auc])

            plt.grid(True)
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
            plt.tight_layout()

            fig_path = os.path.join(layer_graph_dir, f"{file_stub}_BS_{bs}.png")
            plt.savefig(fig_path, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"[SAVED] {fig_path}")

        plot_and_save_ce("Avg_CE_Train", "Average CE_Train", "Avg_CE_Train")
        plot_and_save_ce("Avg_CE_Test", "Average CE_Test", "Avg_CE_Test")

if all_auc_records:
    auc_df = pd.DataFrame(
        all_auc_records,
        columns=["Prune_Layer", "Batch_Size", "Pruning_Percentage", "CE_Type", "AUC"]
    )
    csv_path = os.path.join(AUC_DATA_DIR, "AUC_ALL_BS.csv")
    auc_df.to_csv(csv_path, index=False)
    print(f"[SAVED] {csv_path}")

print("\nAll AUC graphs and combined data saved successfully.")


Processing prune layer: CONV
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_90-100\CONV\Avg_CE_Train_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_90-100\CONV\Avg_CE_Test_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_90-100\CONV\Avg_CE_Train_BS_1024.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_90-100\CONV\Avg_CE_Test_BS_1024.png

Processing prune layer: FHL
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_90-100\FHL\Avg_CE_Train_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_90-100\FHL\Avg_CE_Test_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physl

In [19]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE_DIR_ROOT = r"C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV": "prune_layers_CONV",
    "FHL": "prune_layers_FHL",
    "SHL": "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL": "prune_layers_ALL"
}

BATCH_SIZES = [64, 1024]
PRUNING_PERCENTAGES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.92, 0.94, 0.96, 0.98, 0.985, 0.99, 0.995, 1.0]

# =========================
# OUTPUT
# =========================
AUC_GRAPH_DIR = os.path.join(BASE_DIR_ROOT, "AUC_graph_0-90-100")
AUC_DATA_DIR = os.path.join(BASE_DIR_ROOT, "AUC_data_0-90-100")
os.makedirs(AUC_GRAPH_DIR, exist_ok=True)
os.makedirs(AUC_DATA_DIR, exist_ok=True)

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10
})

CE_MAX = np.log(10)
X_CUTOFF = 400

# =========================
# COLORS
# =========================
PRUNING_COLOR_MAP = {
    0.0:   "#17becf",
    0.1:   "#B9D9EB",
    0.2:   "#bcbd22",
    0.3:   "#7f7f7f",
    0.4:   "#e377c2",
    0.5:   "#8c564b",
    0.6:   "#800080",
    0.7:   "#d62728",
    0.8:   "#2ca02c",
    0.9:   "#C5B0D5",
    0.92:  "#8DD3C7",
    0.94:  "#BEBADA",
    0.96:  "#80B1D3",
    0.98:  "#FDB462",
    0.985: "#6A5ACD",
    0.99:  "#FF6F61",
    0.995: "#00A86B",
    1.0:   "#1f77b4"
}
# =========================
# HELPERS
# =========================
def fmt_p(p):
    s = f"{p:.3f}".rstrip("0").rstrip(".")
    if "." not in s:
        s += ".0"
    return s

def label_from_p(p):
    pct = p * 100
    if abs(pct - round(pct)) < 1e-9:
        return f"P%={int(round(pct))}"
    return f"P%={pct:g}"

all_auc_records = []

# =========================
# MAIN LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:
    print(f"\nProcessing prune layer: {prune_layer}")

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])

    layer_graph_dir = os.path.join(AUC_GRAPH_DIR, prune_layer)
    layer_data_dir = os.path.join(AUC_DATA_DIR, prune_layer)
    os.makedirs(layer_graph_dir, exist_ok=True)
    os.makedirs(layer_data_dir, exist_ok=True)

    for bs in BATCH_SIZES:
        avg_dfs = {}

        for p in PRUNING_PERCENTAGES:
            p_str = fmt_p(p)
            file_path = os.path.join(
                base_dir,
                f"p-percentage_{p_str}",
                f"batch_size_{bs}",
                f"averaged_runs_conv_{prune_layer}_p_{p_str}_bs_{bs}.csv"
            )

            if not os.path.isfile(file_path):
                print(f"[WARNING] Missing file: {file_path}")
                continue

            avg_dfs[p] = pd.read_csv(file_path)

        if not avg_dfs:
            print(f"[WARNING] No data found for {prune_layer}, BS={bs}")
            continue

        non100_dfs = {
            p: df[df["Batch_Number"] <= X_CUTOFF].copy()
            for p, df in avg_dfs.items()
            if p != 1.0 and "Batch_Number" in df.columns
        }
        non100_dfs = {p: df for p, df in non100_dfs.items() if not df.empty}

        if non100_dfs:
            lowest_max_batch = min(df["Batch_Number"].max() for df in non100_dfs.values())
        else:
            all_trunc = {
                p: df[df["Batch_Number"] <= X_CUTOFF].copy()
                for p, df in avg_dfs.items()
                if "Batch_Number" in df.columns
            }
            all_trunc = {p: df for p, df in all_trunc.items() if not df.empty}
            if not all_trunc:
                print(f"[WARNING] No valid Batch_Number data for {prune_layer}, BS={bs}")
                continue
            lowest_max_batch = min(df["Batch_Number"].max() for df in all_trunc.values())

        def plot_and_save_ce(ce_column, y_label, file_stub):
            plt.figure(figsize=(10, 5))
            plt.title(f"CNN-{prune_layer} | FMNIST | BS={bs}")
            plt.xlabel("Batch Number")
            plt.ylabel(y_label)
            plt.xlim(0, lowest_max_batch)
            plt.xticks(np.arange(0, int(lowest_max_batch) + 1, 50))
            plt.yticks(np.arange(0, 2.5, 0.5))
            plt.ylim(0, 2.5)

            plt.text(
                0.40,
                CE_MAX,
                r"$\ln(10)$",
                transform=plt.gca().get_yaxis_transform(),
                fontsize=12,
                va="center",
                ha="left",
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1)
            )

            for p in sorted(avg_dfs.keys(), reverse=True):
                df = avg_dfs[p].copy()
                df = df[df["Batch_Number"] <= min(X_CUTOFF, lowest_max_batch)]

                color = PRUNING_COLOR_MAP.get(p, "#000000")

                if p == 1.0:
                    ce_value = min(df[ce_column].iloc[-1], CE_MAX)
                    x = np.array([0, lowest_max_batch])
                    y = np.array([ce_value, ce_value])
                    auc = ce_value * lowest_max_batch

                    plt.plot(
                        x, y,
                        color=color,
                        linewidth=2.5,
                        label=f"{label_from_p(p)} | AUC={auc:.2f}"
                    )
                else:
                    x = df["Batch_Number"].to_numpy()
                    y = np.minimum(df[ce_column].to_numpy(), CE_MAX)

                    mask = np.isfinite(x) & np.isfinite(y)
                    x, y = x[mask], y[mask]

                    if len(x) == 0:
                        continue

                    auc = np.trapezoid(y, x)

                    plt.plot(
                        x, y,
                        color=color,
                        linewidth=2.0,
                        label=f"{label_from_p(p)} | AUC={auc:.2f}"
                    )
                    plt.fill_between(x, y, alpha=0.2, color=color)

                all_auc_records.append([prune_layer, bs, p, ce_column, auc])

            plt.grid(True)
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
            plt.tight_layout()

            fig_path = os.path.join(layer_graph_dir, f"{file_stub}_BS_{bs}.png")
            plt.savefig(fig_path, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"[SAVED] {fig_path}")

        plot_and_save_ce("Avg_CE_Train", "Average CE_Train", "Avg_CE_Train")
        plot_and_save_ce("Avg_CE_Test", "Average CE_Test", "Avg_CE_Test")

if all_auc_records:
    auc_df = pd.DataFrame(
        all_auc_records,
        columns=["Prune_Layer", "Batch_Size", "Pruning_Percentage", "CE_Type", "AUC"]
    )
    csv_path = os.path.join(AUC_DATA_DIR, "AUC_ALL_BS.csv")
    auc_df.to_csv(csv_path, index=False)
    print(f"[SAVED] {csv_path}")

print("\nAll AUC graphs and combined data saved successfully.")


Processing prune layer: CONV
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_0-90-100\CONV\Avg_CE_Train_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_0-90-100\CONV\Avg_CE_Test_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_0-90-100\CONV\Avg_CE_Train_BS_1024.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_0-90-100\CONV\Avg_CE_Test_BS_1024.png

Processing prune layer: FHL
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_0-90-100\FHL\Avg_CE_Train_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST\AUC_graph_0-90-100\FHL\Avg_CE_Test_BS_64.png
[SAVED] C:\Users\Student\Desktop\Neural_re